# Inspect Extracted Images From R2

This notebook previews image-extraction outputs that already exist in R2.
It does not rerun Newspaper Navigator extraction. Instead, it lists uploaded
metadata files, reads one or more `regions.jsonl` files, downloads a few crop
PNGs locally, and displays them for inspection.

## Setup

Before running these cells:

```bash
conda env create -f img_extractor.yml
conda activate image-extractor
```

The notebook expects your normal R2 environment variables to be available,
either in `.env` at repo root or exported in the shell.

In [ ]:
from collections import Counter
from pathlib import Path
import json

from dotenv import load_dotenv
from IPython.display import display
from PIL import Image

from r2_utils import (
    DEFAULT_R2_PREFIX,
    build_r2_client,
    download_r2_object,
    list_r2_keys,
    normalized_prefix,
    read_r2_jsonl,
)

load_dotenv(Path("..").resolve() / ".env")

client, bucket = build_r2_client()
r2_prefix = DEFAULT_R2_PREFIX
metadata_prefix = f"{normalized_prefix(r2_prefix)}extracted_images/"
preview_dir = Path("../output/r2_image_preview")
preview_dir.mkdir(parents=True, exist_ok=True)

print("Bucket:", bucket)
print("Metadata prefix:", metadata_prefix)
print("Preview dir:", preview_dir.resolve())

## Discover Uploaded Documents

Each processed document should have a `regions.jsonl` file under:

- `archive/extracted_images/<year>/<month>/<doc_id>/regions.jsonl`

In [ ]:
metadata_keys = list(list_r2_keys(client, bucket, metadata_prefix, suffix="regions.jsonl"))
print(f"Found {len(metadata_keys)} processed documents")
metadata_keys[:10]

In [ ]:
# Pick one metadata file to inspect. You can replace the index or hardcode a key.
sample_metadata_key = metadata_keys[0]
sample_metadata_key

## Load One Document's Regions

In [ ]:
rows = read_r2_jsonl(client, bucket, sample_metadata_key)
print(f"Loaded {len(rows)} regions from {sample_metadata_key}")
rows[:3]

In [ ]:
label_counts = Counter(row["label"] for row in rows)
label_counts

## Download And Preview A Few Crops

In [ ]:
for row in rows[:6]:
    crop_key = row["r2_crop_key"]
    local_path = preview_dir / Path(crop_key).name
    download_r2_object(client, bucket, crop_key, local_path)
    print(json.dumps({
        "doc_id": row["doc_id"],
        "page_number": row["page_number"],
        "label": row["label"],
        "score": row["score"],
        "r2_crop_key": crop_key,
    }, indent=2))
    display(Image.open(local_path))

## Inspect Multiple Documents

This cell samples several `regions.jsonl` files and summarizes label counts across them.

In [ ]:
sample_size = min(25, len(metadata_keys))
aggregate_counter = Counter()
total_regions = 0

for key in metadata_keys[:sample_size]:
    doc_rows = read_r2_jsonl(client, bucket, key)
    aggregate_counter.update(row["label"] for row in doc_rows)
    total_regions += len(doc_rows)

print(f"Sampled {sample_size} metadata files")
print(f"Total sampled regions: {total_regions}")
aggregate_counter